In [1]:
import requests
import json
import pandas as pd
import copy
import time
import requests
import pandas as pd
import json
import copy
import time

In [2]:
def generar_meses():
    meses = ["ene", "feb", "mar", "abr", "may", "jun",
             "jul", "ago", "sep", "oct", "nov", "dic"]
    
    resultado = []
    for anio in range(2015, 2026):
        for mes in meses:
            resultado.append(f"{mes}. {str(anio)[-2:]}")
    
    return resultado


def parse_looker_response(data, mes):
    dataset = data['dataResponse'][0]['dataSubset'][0]['dataset']['tableDataset']
    
    columns_list = dataset['column']
    total_rows = dataset['size']

    final_columns = {}

    for i, col_data in enumerate(columns_list):

        col_name = col_data.get('name') or f"col_{i}"

        key = 'stringColumn' if 'stringColumn' in col_data else 'doubleColumn'
        values = col_data.get(key, {}).get('values', [])
        null_indices = col_data.get('nullIndex', [])

        full_col = []
        val_ptr = 0

        for idx in range(total_rows):
            if idx in null_indices:
                full_col.append(None)
            else:
                full_col.append(values[val_ptr])
                val_ptr += 1

        final_columns[col_name] = full_col

    df = pd.DataFrame(final_columns)
    df["mes"] = mes

    return df


# 🔥 NUEVA FUNCIÓN
def find_mes_filter(payload):
    filters = payload['dataRequest'][0]['datasetSpec']['filters']
    
    for i, f in enumerate(filters):
        try:
            exp = f['filterDefinition']['filterExpression']
            
            if 'stringValues' in exp:
                val = exp['stringValues'][0]
                
                # detecta formato tipo "ene. 24"
                if isinstance(val, str) and "." in val:
                    return i
        except:
            continue
    
    raise ValueError("No se encontró el filtro de mes")


def run_looker_pipeline(
    cookies, 
    headers, 
    params, 
    json_data,
    pais=None,
    ciudad=None,
    operacion=None,
    columnas=None,
    mes_filter_idx=None,        # 🔥 NUEVO
    mes_field_name=None         # 🔥 OPCIONAL (ej: "_fecha_")
):
    
    meses = generar_meses()
    dfs = []

    # ==============================
    # 🔥 DETECCIÓN DEL FILTRO
    # ==============================
    if mes_filter_idx is None:
        filters = json_data['dataRequest'][0]['datasetSpec']['filters']
        
        for i, f in enumerate(filters):
            try:
                field = f['filterDefinition']['filterExpression']['queryTimeTransformation']['dataTransformation']['sourceFieldName']
                
                if mes_field_name and field == mes_field_name:
                    mes_filter_idx = i
                    break
            except:
                continue

        if mes_filter_idx is None:
            raise ValueError("No se encontró el filtro de mes")

    print(f"📍 Usando filtro de mes en índice: {mes_filter_idx}")

    # ==============================
    # LOOP
    # ==============================
    for mes in meses:
        try:
            payload = copy.deepcopy(json_data)

            payload['dataRequest'][0]['datasetSpec']['filters'][mes_filter_idx]['filterDefinition']['filterExpression']['stringValues'] = [mes]

            response = requests.post(
                'https://lookerstudio.google.com/embed/u/0/batchedDataV2',
                params=params,
                cookies=cookies,
                headers=headers,
                json=payload,
            )

            if response.status_code != 200:
                print(f"❌ Error en {mes}")
                continue

            raw_str = response.text[5:].strip()
            data = json.loads(raw_str)

            df = parse_looker_response(data, mes)

            if len(df) == 0:
                print(f"⚠️ {mes} sin datos")
                continue

            # ==============================
            # METADATOS
            # ==============================
            if pais is not None:
                df["pais"] = pais
            if ciudad is not None:
                df["ciudad"] = ciudad
            if operacion is not None:
                df["operacion"] = operacion

            # ==============================
            # RENOMBRE
            # ==============================
            if columnas is not None:
                df = df.rename(columns=columnas)

            dfs.append(df)

            print(f"✅ {mes} | {len(df)} filas")

            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error en {mes}: {e}")

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()

### Buenos Aires

#### Venta

In [3]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AIajjRpatlPZWVHlfVee4OU5CqCZg:1774569719639',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=i36aysuCt0eKKwYfNKty0MRnM5iU0tDqNIta3A219XwwPEsSCn0-7Dj_J-sfCWf7-M5hN9BcxJOCAGLsk63j-oJLNVHs2FP_IZA3jYJ-3xIAEBpoVA5cJe0X2px5M1fSm7LrUhydHP0nc-e4avG--YXOYoc1pSxO-LUlLMhGgsCIO8wPYj1BTNSvgu0r8nSsaFSnm2x4H69edtzlFUADMltgTuuXe4HoPJLQYvEB-uJQLoM9mL-BSDDic8SmxTiqFE4mG0UaRqpoAqSqLRpNv7s-sbqtGsEzXmXEeRRpNesJ7XMMHl5h5_LjB3MZknTaqknfF9gkq4cdsytDmmYmWtKVuH31cBfLdC649LgCfnpFrBA9tF-yncht58dowjk1mj86bxuUyyruc5f-j4gjygTwBQO4uI4-zFDXdQWOXRfwMC3Fnw8qCb4b9i6wNjpvaTniev7QqlxP4QO16V24YYMWC99qRhd7wWraeL0d1Is15Bimaqgn-YrMSDSlEHfZ1GvytBTPkiLdkYeBl3mCXs08hPd-kH_O4FCg59RWIuEa6HjeRxuKuziamGGhOuZVQRG31njfeb8baowuBq0xfhL0IG5wOAS-W64CIRKWKdy_uYoXQubuqqFVvEk_GU7eQKRQHvqdEi9VuUkm2bcxYclaRQdGPuBxzPqUOxQp3TLmLgCeM2Zu6n4o-vbyDTs667XNshUdKgb47OCkVb-MI1t5IuOCjJ7Y7nT19WdpckMA861wswy6Mvh7k-arDw86YkHqvQFaOjGDuRCzVOubp2TcZ16rFO9T-c5KtRaFhAvISa94p1Tod1tIs-HTFrmVMlvbHBfjWd3Ak9wVvKFww28W20nXlBr7-1qITaep_uilWTU4wIR7ANiAgUJNrf61riP9a1CyImtC3aaMxa0DJ1BZq0gF6Zz8GpgInsRsl6CKQpAKAzEqN0RoLg5mccKctC_bZNdI1yXK3p8W8PsIRwhOMXY-ynXjqFU_4Aae8hKR3Gx657EXb_yjwlls-am2g7bZOtEDRB4WjznCk8a4tGMdn5a3V_4zSdA6KBbDwZcJRmpLNser9DPnhYfChjS1T8BZD_-mt_vSXNb1QARWuyHeE47XN5C15BhVrlKbo57RQ_OQzQ9xBHNkbdypnLLI-7j0jPvrwP2WT0rdgUgfpqFgNIh8OwIOG2ycoCDSUaqwIbyvKIjweYuH6g3qRz4Fm3Ag-TiyMb6u5lzm2PkvN_YvSzNeo0T89wej-m_tcJf_kQCTqd8REVb7boCEfdoXDLhDiKt-s45B4MsPtO6px6hAENFgUqNUBLENt869pgPFK-LhZu40QEyfu_UylgaQAHsXl3gbK1RzIsKwqFxNSKWsA7KL4aF7i7cAmTEmEhqBBU-6OvVnmHOW7jLcudl4V5BiteAl-3n0qEabYTLxp8EKXOdnsgtpTtMrdqirzVU7XMJLv3h0TdLBJX0hwnSL21rDw69ysiZYdhpWQMgPWXxhnP_CqFCwn0jGwiRNU0friM7VbERzwUEJK0oJss2Bg_6zNlWzYuoeAXSctM67cDjt8sWfljH7QeAqzJiq3TZ_TRJ3dEWPLUNWw_pjU8XBZGy2imKlgaRv7f9ENud3a5inxv7DMib_1DJx25Sf-_NY7R5vcVsLasv4fP8fejnGdHZ-tbiUxbPCDtE1WsCMLkPNEH60dk_P9xyj7luMGQwRvAzYIuqTkFO99kk_82pSjltUz50XEfiYp3oug57AM2rjTMoXOzLVnqG7X1f9BLqpg6WaVlk2odP1z_UaSFXRFNIE__NaJOXFo620t4Q8EtkwRo88mI22_-cKOiz-_w4QnZ-p1jkYk69q1IWs4GYhVTCDlYpAD2yr5h7w6g_C-AIaOx-AtVxD6p8Lbe2s--yl9HmGjImyzFO562QsYuL7_2H38xCyL4k9xLEwWw1wA8smeAmKJ3xi8KTiiDb1drJwegyGYAxbAonbFVeqUKONM9Fuv_q4VbjmnXheEO6zSnonWH2tB1khekvUhbnWLS1ka0sQ-aO-xfqVh_SWxN0BSB8Hz6KCbMY1RBXuy3X9it2_fSl9UjX0wSM_uc5pxhPYfpaCEWI_Qbuw3_wjMDuyrbCh3i_gCJ2cJIi37roKzseR1JYqqykGko_D',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCSiiiio71K_05-e7d2kd0Ee58CaYAIe0jVAeS8idlV6Hft0avvVZz1lMvfbfEAA',
    '__Secure-3PSIDCC': 'AKEyXzVEitg5fgc-lbN6BPoLvjoBIrJ8aLMkFlfJEdL3GCRbucCo1Xy9iKUFA7tBmbOEMN5MZctY9g',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/4c3f2f77-8384-4b7d-995b-5f3317386432/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AIajjRpatlPZWVHlfVee4OU5CqCZg:1774569719639',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AIajjRpatlPZWVHlfVee4OU5CqCZg:1774569719639; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=i36aysuCt0eKKwYfNKty0MRnM5iU0tDqNIta3A219XwwPEsSCn0-7Dj_J-sfCWf7-M5hN9BcxJOCAGLsk63j-oJLNVHs2FP_IZA3jYJ-3xIAEBpoVA5cJe0X2px5M1fSm7LrUhydHP0nc-e4avG--YXOYoc1pSxO-LUlLMhGgsCIO8wPYj1BTNSvgu0r8nSsaFSnm2x4H69edtzlFUADMltgTuuXe4HoPJLQYvEB-uJQLoM9mL-BSDDic8SmxTiqFE4mG0UaRqpoAqSqLRpNv7s-sbqtGsEzXmXEeRRpNesJ7XMMHl5h5_LjB3MZknTaqknfF9gkq4cdsytDmmYmWtKVuH31cBfLdC649LgCfnpFrBA9tF-yncht58dowjk1mj86bxuUyyruc5f-j4gjygTwBQO4uI4-zFDXdQWOXRfwMC3Fnw8qCb4b9i6wNjpvaTniev7QqlxP4QO16V24YYMWC99qRhd7wWraeL0d1Is15Bimaqgn-YrMSDSlEHfZ1GvytBTPkiLdkYeBl3mCXs08hPd-kH_O4FCg59RWIuEa6HjeRxuKuziamGGhOuZVQRG31njfeb8baowuBq0xfhL0IG5wOAS-W64CIRKWKdy_uYoXQubuqqFVvEk_GU7eQKRQHvqdEi9VuUkm2bcxYclaRQdGPuBxzPqUOxQp3TLmLgCeM2Zu6n4o-vbyDTs667XNshUdKgb47OCkVb-MI1t5IuOCjJ7Y7nT19WdpckMA861wswy6Mvh7k-arDw86YkHqvQFaOjGDuRCzVOubp2TcZ16rFO9T-c5KtRaFhAvISa94p1Tod1tIs-HTFrmVMlvbHBfjWd3Ak9wVvKFww28W20nXlBr7-1qITaep_uilWTU4wIR7ANiAgUJNrf61riP9a1CyImtC3aaMxa0DJ1BZq0gF6Zz8GpgInsRsl6CKQpAKAzEqN0RoLg5mccKctC_bZNdI1yXK3p8W8PsIRwhOMXY-ynXjqFU_4Aae8hKR3Gx657EXb_yjwlls-am2g7bZOtEDRB4WjznCk8a4tGMdn5a3V_4zSdA6KBbDwZcJRmpLNser9DPnhYfChjS1T8BZD_-mt_vSXNb1QARWuyHeE47XN5C15BhVrlKbo57RQ_OQzQ9xBHNkbdypnLLI-7j0jPvrwP2WT0rdgUgfpqFgNIh8OwIOG2ycoCDSUaqwIbyvKIjweYuH6g3qRz4Fm3Ag-TiyMb6u5lzm2PkvN_YvSzNeo0T89wej-m_tcJf_kQCTqd8REVb7boCEfdoXDLhDiKt-s45B4MsPtO6px6hAENFgUqNUBLENt869pgPFK-LhZu40QEyfu_UylgaQAHsXl3gbK1RzIsKwqFxNSKWsA7KL4aF7i7cAmTEmEhqBBU-6OvVnmHOW7jLcudl4V5BiteAl-3n0qEabYTLxp8EKXOdnsgtpTtMrdqirzVU7XMJLv3h0TdLBJX0hwnSL21rDw69ysiZYdhpWQMgPWXxhnP_CqFCwn0jGwiRNU0friM7VbERzwUEJK0oJss2Bg_6zNlWzYuoeAXSctM67cDjt8sWfljH7QeAqzJiq3TZ_TRJ3dEWPLUNWw_pjU8XBZGy2imKlgaRv7f9ENud3a5inxv7DMib_1DJx25Sf-_NY7R5vcVsLasv4fP8fejnGdHZ-tbiUxbPCDtE1WsCMLkPNEH60dk_P9xyj7luMGQwRvAzYIuqTkFO99kk_82pSjltUz50XEfiYp3oug57AM2rjTMoXOzLVnqG7X1f9BLqpg6WaVlk2odP1z_UaSFXRFNIE__NaJOXFo620t4Q8EtkwRo88mI22_-cKOiz-_w4QnZ-p1jkYk69q1IWs4GYhVTCDlYpAD2yr5h7w6g_C-AIaOx-AtVxD6p8Lbe2s--yl9HmGjImyzFO562QsYuL7_2H38xCyL4k9xLEwWw1wA8smeAmKJ3xi8KTiiDb1drJwegyGYAxbAonbFVeqUKONM9Fuv_q4VbjmnXheEO6zSnonWH2tB1khekvUhbnWLS1ka0sQ-aO-xfqVh_SWxN0BSB8Hz6KCbMY1RBXuy3X9it2_fSl9UjX0wSM_uc5pxhPYfpaCEWI_Qbuw3_wjMDuyrbCh3i_gCJ2cJIi37roKzseR1JYqqykGko_D; __Secure-3PSIDTS=sidts-CjEBWhotCSiiiio71K_05-e7d2kd0Ee58CaYAIe0jVAeS8idlV6Hft0avvVZz1lMvfbfEAA; __Secure-3PSIDCC=AKEyXzVEitg5fgc-lbN6BPoLvjoBIrJ8aLMkFlfJEdL3GCRbucCo1Xy9iKUFA7tBmbOEMN5MZctY9g',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '4c3f2f77-8384-4b7d-995b-5f3317386432',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '6e7fd983-2bf0-4669-97b2-b3029cb6c157',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_t0kqnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_i13pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_3k5pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_pozo_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_4k5pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_xc6pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_4k5pnc7xgc',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 100,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'CABA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_hustnc7xgc',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'dic. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_mfqqnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_id_n3_',
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [4]:
ba_venta = run_looker_pipeline(cookies, headers, params, json_data, pais="Argentina",ciudad="Buenos Aires",operacion="sell", 
                                mes_field_name="_fecha_",
                                 columnas={"col_0":"neighborhood","col_1":"new","col_2": "index","col_3": "used"})
print("Total de registros", len(ba_venta))
ba_venta.to_csv("ba_venta.csv", encoding="latin1",index=False)
ba_venta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
✅ dic. 15 | 47 filas
✅ ene. 16 | 47 filas
✅ feb. 16 | 47 filas
✅ mar. 16 | 47 filas
✅ abr. 16 | 47 filas
✅ may. 16 | 47 filas
✅ jun. 16 | 47 filas
✅ jul. 16 | 47 filas
✅ ago. 16 | 47 filas
✅ sep. 16 | 47 filas
✅ oct. 16 | 47 filas
✅ nov. 16 | 47 filas
✅ dic. 16 | 47 filas
✅ ene. 17 | 47 filas
✅ feb. 17 | 47 filas
✅ mar. 17 | 47 filas
✅ abr. 17 | 47 filas
✅ may. 17 | 47 filas
✅ jun. 17 | 47 filas
✅ jul. 17 | 47 filas
✅ ago. 17 | 47 filas
✅ sep. 17 | 47 filas
✅ oct. 17 | 47 filas
✅ nov. 17 | 47 filas
✅ dic. 17 | 47 filas
✅ ene. 18 | 47 filas
✅ feb. 18 | 47 filas
✅ mar. 18 | 47 filas
✅ abr. 18 | 47 filas
✅ may. 18 | 47 filas
✅ jun. 18 | 47 filas
✅ jul. 18 | 47 filas
✅ ago. 18 | 47 filas
✅ sep. 18 | 47 filas
✅ oct. 18 | 47 fila

C:\Users\claud\AppData\Local\Temp\ipykernel_43000\3211343472.py:160: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True)


,neighborhood,new,index,used,col_4,mes,pais,ciudad,operacion
0,Boedo,NaN,NaN,None,None,dic. 15,Argentina,Buenos Aires,rent
1,Parque Patricios,NaN,NaN,None,None,dic. 15,Argentina,Buenos Aires,rent
2,Balvanera,NaN,NaN,None,None,dic. 15,Argentina,Buenos Aires,rent
3,Liniers,NaN,NaN,None,None,dic. 15,Argentina,Buenos Aires,rent
4,Chacarita,NaN,NaN,None,None,dic. 15,Argentina,Buenos Aires,rent
...,...,...,...,...,...,...,...,...,...
5307,Villa Riachuelo,1722.0,NaN,1608,1570,dic. 25,Argentina,Buenos Aires,rent
5308,Parque Avellaneda,1837.0,1900.0,1580,1434,dic. 25,Argentina,Buenos Aires,rent
5309,La Boca,2230.0,2364.0,1535,1460,dic. 25,Argentina,Buenos Aires,rent
5310,Nueva Pompeya,1956.0,NaN,1473,1319,dic. 25,Argentina,Buenos Aires,rent


#### Buenos Aires 
##### renta

In [6]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AJT81fQ9lnlXF4Xz3SW1mrwCnFoYA:1774571254621',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCV3Ipd6ZhdMCU9UyB2NS-KMxav-PTufS08Rf9PLfxMmbxktEqBf3A-wYWcDoEAA',
    '__Secure-3PSIDCC': 'AKEyXzUHhJ6LabhQ6QOpfSm_TEDdi0uZu5crbpK9NfFjL21bkjuif6u8TZ40MJN33QODDCxS3Irmnw',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/17f8fee8-1a1e-42e2-b20a-9e9f2350ac28/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AJT81fQ9lnlXF4Xz3SW1mrwCnFoYA:1774571254621',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AJT81fQ9lnlXF4Xz3SW1mrwCnFoYA:1774571254621; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD; __Secure-3PSIDTS=sidts-CjEBWhotCV3Ipd6ZhdMCU9UyB2NS-KMxav-PTufS08Rf9PLfxMmbxktEqBf3A-wYWcDoEAA; __Secure-3PSIDCC=AKEyXzUHhJ6LabhQ6QOpfSm_TEDdi0uZu5crbpK9NfFjL21bkjuif6u8TZ40MJN33QODDCxS3Irmnw',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '17f8fee8-1a1e-42e2-b20a-9e9f2350ac28',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': 'd37cc422-d708-4174-9c1d-4253dac1dd5a',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_k2g8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_pfc8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_j7c8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_czd8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_j7c8446xgc',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 100,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'CABA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_wwga546xgc',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'ene. 26',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_tvr8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_id_n3_',
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [8]:
ba_renta = run_looker_pipeline(cookies, headers, params, json_data, pais="Argentina",ciudad="Buenos Aires",operacion="rent",  
                                mes_field_name="_fecha_",
                                 columnas={
                                                                                                                                     "col_0":"neighborhood",
                                                                                                                                     "col_1": "new",
                                                                                                                                     "col_2": "preconstruction",
                                                                                                                                     "col_3": "index",
                                                                                                                                     "col_4": "used"  
                                                                                                                                     })
print("Total de registros", len(ba_renta))
ba_renta.to_csv("ba_renta.csv", encoding="latin1",index=False)
ba_renta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
✅ dic. 15 | 45 filas
✅ ene. 16 | 45 filas
✅ feb. 16 | 43 filas
✅ mar. 16 | 43 filas
✅ abr. 16 | 46 filas
✅ may. 16 | 45 filas
✅ jun. 16 | 45 filas
✅ jul. 16 | 46 filas
✅ ago. 16 | 44 filas
✅ sep. 16 | 45 filas
✅ oct. 16 | 45 filas
✅ nov. 16 | 44 filas
✅ dic. 16 | 45 filas
✅ ene. 17 | 46 filas
✅ feb. 17 | 45 filas
✅ mar. 17 | 45 filas
✅ abr. 17 | 45 filas
✅ may. 17 | 44 filas
✅ jun. 17 | 45 filas
✅ jul. 17 | 45 filas
✅ ago. 17 | 46 filas
✅ sep. 17 | 44 filas
✅ oct. 17 | 44 filas
✅ nov. 17 | 45 filas
✅ dic. 17 | 45 filas
✅ ene. 18 | 45 filas
✅ feb. 18 | 45 filas
✅ mar. 18 | 45 filas
✅ abr. 18 | 46 filas
✅ may. 18 | 45 filas
✅ jun. 18 | 46 filas
✅ jul. 18 | 46 filas
✅ ago. 18 | 46 filas
✅ sep. 18 | 46 filas
✅ oct. 18 | 46 fila

,neighborhood,new,preconstruction,index,mes,pais,ciudad,operacion
0,San Nicolas,39682.0,37313.0,36282.0,dic. 15,Argentina,Buenos Aires,rent
1,Nueva Pompeya,NaN,24191.0,18963.0,dic. 15,Argentina,Buenos Aires,rent
2,Palermo,25052.0,23342.0,23116.0,dic. 15,Argentina,Buenos Aires,rent
3,Recoleta,27642.0,23126.0,22715.0,dic. 15,Argentina,Buenos Aires,rent
4,Colegiales,26701.0,21854.0,21464.0,dic. 15,Argentina,Buenos Aires,rent
...,...,...,...,...,...,...,...,...
4841,Liniers,808974.0,664302.0,644596.0,dic. 25,Argentina,Buenos Aires,rent
4842,Parque Avellaneda,NaN,651689.0,639060.0,dic. 25,Argentina,Buenos Aires,rent
4843,Floresta,NaN,649624.0,644929.0,dic. 25,Argentina,Buenos Aires,rent
4844,Constitución,728052.0,640507.0,626322.0,dic. 25,Argentina,Buenos Aires,rent


#### Córdoba 
#### Venta

In [9]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AKED93E42rSittNrmNuz5Alir4wfA:1774571685883',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCXFn0briAfgN9IWJr8GjBRY_WzdLEv_1PqCcNWNpXQc_T1sCdgCH3ToeS7bvEAA',
    '__Secure-3PSIDCC': 'AKEyXzUQm624jrQtyzByZjO0a3FSwY0qjUfdaRGhrek7eJT3L1QqDzodgSASVJN6Vr_gz0jS93TnIA',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/340bc6fc-3b47-4468-8182-5f2601bb516e/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AKED93E42rSittNrmNuz5Alir4wfA:1774571685883',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AKED93E42rSittNrmNuz5Alir4wfA:1774571685883; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD; __Secure-3PSIDTS=sidts-CjEBWhotCXFn0briAfgN9IWJr8GjBRY_WzdLEv_1PqCcNWNpXQc_T1sCdgCH3ToeS7bvEAA; __Secure-3PSIDCC=AKEyXzUQm624jrQtyzByZjO0a3FSwY0qjUfdaRGhrek7eJT3L1QqDzodgSASVJN6Vr_gz0jS93TnIA',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '340bc6fc-3b47-4468-8182-5f2601bb516e',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '6e7fd983-2bf0-4669-97b2-b3029cb6c157',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_t0kqnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_i13pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_3k5pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_pozo_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_4k5pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_xc6pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_4k5pnc7xgc',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 100,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'CBA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_hustnc7xgc',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'ago. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_mfqqnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_id_n3_',
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [10]:
cor_venta = run_looker_pipeline(cookies, headers, params, json_data, pais="Argentina",ciudad="Cordoba",operacion="sell",
                                
                                mes_field_name="_fecha_",
                                 columnas={
                                                                                                                                     "col_0":"neighborhood",
                                                                                                                                     "col_1": "new",
                                                                                                                                     "col_2": "preconstruction",
                                                                                                                                     "col_3": "index",
                                                                                                                                     "col_4": "used"  
                                                                                                                                     })
print("Total de registros", len(cor_venta))
cor_venta.to_csv("cor_venta.csv", encoding="latin1",index=False)
cor_venta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,preconstruction,index,used,mes,pais,ciudad,operacion
0,CERRO CHICO,NaN,NaN,2852,2786.0,may. 21,Argentina,Cordoba,sell
1,COUNTRY JOCKEY CLUB,2271.0,NaN,2177,NaN,may. 21,Argentina,Cordoba,sell
2,TEJAS DEL SUR,NaN,NaN,1960,NaN,may. 21,Argentina,Cordoba,sell
3,CERRO DE LAS ROSAS,NaN,NaN,1910,1859.0,may. 21,Argentina,Cordoba,sell
4,JARDIN DEL SUD,2093.0,NaN,1870,1594.0,may. 21,Argentina,Cordoba,sell
...,...,...,...,...,...,...,...,...,...
2466,VALLE CERCANO,692.0,NaN,689,NaN,nov. 25,Argentina,Cordoba,sell
2467,SAN VICENTE,596.0,NaN,667,682.0,nov. 25,Argentina,Cordoba,sell
2468,GENERAL BUSTOS,NaN,NaN,638,638.0,nov. 25,Argentina,Cordoba,sell
2469,MARQUES DE SOBREMONTE,NaN,NaN,565,565.0,nov. 25,Argentina,Cordoba,sell


#### Córdoba 
#### Renta

In [11]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AJuJ_v9853dI_z29X8_ayJ43AlPmg:1774572019415',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCXFn0briAfgN9IWJr8GjBRY_WzdLEv_1PqCcNWNpXQc_T1sCdgCH3ToeS7bvEAA',
    '__Secure-3PSIDCC': 'AKEyXzWrlmaN79CHBQKKTilE--Ady6Nx7-5COmZ7qkh7HMVI67nS8g8rr9J8ddekzDZ1Ey-CQp1DhQ',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/ee4d13ac-fe80-4476-a2c0-5cbb51fe7601/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AJuJ_v9853dI_z29X8_ayJ43AlPmg:1774572019415',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AJuJ_v9853dI_z29X8_ayJ43AlPmg:1774572019415; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD; __Secure-3PSIDTS=sidts-CjEBWhotCXFn0briAfgN9IWJr8GjBRY_WzdLEv_1PqCcNWNpXQc_T1sCdgCH3ToeS7bvEAA; __Secure-3PSIDCC=AKEyXzWrlmaN79CHBQKKTilE--Ady6Nx7-5COmZ7qkh7HMVI67nS8g8rr9J8ddekzDZ1Ey-CQp1DhQ',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': 'ee4d13ac-fe80-4476-a2c0-5cbb51fe7601',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': 'd37cc422-d708-4174-9c1d-4253dac1dd5a',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_k2g8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_pfc8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_j7c8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_czd8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_j7c8446xgc',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 100,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'CBA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_wwga546xgc',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'ago. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_tvr8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_id_n3_',
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [12]:
cor_renta = run_looker_pipeline(cookies, headers, params, json_data, pais="Argentina",ciudad="Cordoba",operacion="rent",
     mes_field_name="_fecha_",
                                 columnas={
                                                                                                                                     "col_0":"neighborhood",
                                                                                                                                     "col_1": "new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"  
                                                                                                                                     })
print("Total de registros", len(cor_renta))
cor_renta.to_csv("cor_renta.csv", encoding="latin1",index=False)
cor_renta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,QUEBRADA DE LAS ROSAS,NaN,21254,NaN,may. 21,Argentina,Cordoba,rent
1,JARDIN,NaN,19620,NaN,may. 21,Argentina,Cordoba,rent
2,GENERAL PAZ,NaN,19551,19574.0,may. 21,Argentina,Cordoba,rent
3,NUEVA CORDOBA,20193.0,19357,19282.0,may. 21,Argentina,Cordoba,rent
4,GENERAL PUEYRREDON,16496.0,18199,18595.0,may. 21,Argentina,Cordoba,rent
...,...,...,...,...,...,...,...,...
538,MIRADOR DEL CHATEAU,NaN,423725,NaN,nov. 25,Argentina,Cordoba,rent
539,SAN VICENTE,NaN,418376,414825.0,nov. 25,Argentina,Cordoba,rent
540,SAN MARTIN,NaN,401755,392789.0,nov. 25,Argentina,Cordoba,rent
541,LAS PALMAS,NaN,368398,374582.0,nov. 25,Argentina,Cordoba,rent


### Rosario
#### Venta

In [ ]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AJUIgsaD-yS-d8yP6TAEXLBkOHjRQ:1774572405818',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCaRoXoNXQh_LpULm0aselCovonNz6w8uDiOhS_MBXfxyhPXJPHBAz75Ze2MCEAA',
    '__Secure-3PSIDCC': 'AKEyXzUQS2_PzR4DV3IGgkduCncj9WbOh5_1GCbpZ8SMwqFtR4x--9JmFTkYPWxdW4v8WNvn4Ruckg',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/90c89422-6841-451f-8caa-2460fe8a4020/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AJUIgsaD-yS-d8yP6TAEXLBkOHjRQ:1774572405818',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AJUIgsaD-yS-d8yP6TAEXLBkOHjRQ:1774572405818; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD; __Secure-3PSIDTS=sidts-CjEBWhotCaRoXoNXQh_LpULm0aselCovonNz6w8uDiOhS_MBXfxyhPXJPHBAz75Ze2MCEAA; __Secure-3PSIDCC=AKEyXzUQS2_PzR4DV3IGgkduCncj9WbOh5_1GCbpZ8SMwqFtR4x--9JmFTkYPWxdW4v8WNvn4Ruckg',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '90c89422-6841-451f-8caa-2460fe8a4020',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '6e7fd983-2bf0-4669-97b2-b3029cb6c157',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_t0kqnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_i13pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_3k5pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_pozo_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_4k5pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_xc6pnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_4k5pnc7xgc',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 100,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'RSA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_hustnc7xgc',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'jul. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_mfqqnc7xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_id_n3_',
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [13]:
rosario_venta = run_looker_pipeline(
    cookies, headers, params, json_data,
    pais="Argentina",
    ciudad="Rosario",
    operacion="sell",
    columnas={
        "col_0": "neighborhood",
        "col_1": "new",
        "col_2": "preconstruction",
        "col_3": "index",
        "col_4": "used"
    },
     mes_field_name="_fecha_"
)
print("Total de registros", len(rosario_venta))
rosario_venta.to_csv("rosario_venta.csv", encoding="latin1",index=False)
rosario_venta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,preconstruction,index,mes,pais,ciudad,operacion
0,QUEBRADA DE LAS ROSAS,NaN,21254,NaN,may. 21,Argentina,Rosario,sell
1,JARDIN,NaN,19620,NaN,may. 21,Argentina,Rosario,sell
2,GENERAL PAZ,NaN,19551,19574.0,may. 21,Argentina,Rosario,sell
3,NUEVA CORDOBA,20193.0,19357,19282.0,may. 21,Argentina,Rosario,sell
4,GENERAL PUEYRREDON,16496.0,18199,18595.0,may. 21,Argentina,Rosario,sell
...,...,...,...,...,...,...,...,...
538,MIRADOR DEL CHATEAU,NaN,423725,NaN,nov. 25,Argentina,Rosario,sell
539,SAN VICENTE,NaN,418376,414825.0,nov. 25,Argentina,Rosario,sell
540,SAN MARTIN,NaN,401755,392789.0,nov. 25,Argentina,Rosario,sell
541,LAS PALMAS,NaN,368398,374582.0,nov. 25,Argentina,Rosario,sell


In [14]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1ALceDCq5-_QxYUgBZ5VEsg1Bm0rpA:1774572541754',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCaRoXoNXQh_LpULm0aselCovonNz6w8uDiOhS_MBXfxyhPXJPHBAz75Ze2MCEAA',
    '__Secure-3PSIDCC': 'AKEyXzWS60mAz2wOliq5lBo3YVICO72YOGF2beIf30MItp8injC0wemHt_WKXKE27iMOt2vNUpTrfw',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/e79c2a68-c9c9-4701-bda9-f3351fd299f4/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1ALceDCq5-_QxYUgBZ5VEsg1Bm0rpA:1774572541754',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1ALceDCq5-_QxYUgBZ5VEsg1Bm0rpA:1774572541754; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=DX31d-KjbnzzrnX8irOFpBfR1g2pRr7lsbFk3nYFwi8XDeUEr0JUfwQkhtnCsPR2CocVrP-vMBxcD3ywL0sU-9o4yhzo5wVHevUjHDe0nxdl7or7RkwHXnwot7odjdnoVe_UbmkyHGb9ennvkXEZUCBjqouD7mQOV0SDRGEjoA9Baki9lb6NFE2UDU85d23i1lSoW-xhQx4u1QXOFMRfeJw9ipkb24hZJrjM_VnvoJy0S64rptHOb24IE5QTno989X5Ygj2eTbuz0_vKdbF_8-Hd-eQ5K_WPdOSqXjRBUi4kIjA-YBt4OsoQ_H_DFDkGD7EB3x9L0_BUxGRoluO8MP634UcerjxO-JdW4s7clPfbZz67uLAAM1nOeg9EC_q6o4UHnWkTdBr3mZewMjLeQO4QgcxqW-P_ssAMfcU_Rjzvxe9062AwkuQ0ua4VKRHCnBLwx__FjOLyhq0AKI7govLZrOvSxcWgJx_aSYbK9X0k_sTyx7IufFcKgZvjxprfmkPIpdYr-ryCh1o0_hd6YH7MwOrHIvrBr7AvxHeudKhdgOaCvKB6kkU1LyhiqXqStX-TzTYoiKRy0FoC9lSfv8u2jgi4-IJmwpoUHJE70-G_wjOJQ_WcXLrUxpuHWaUGsNjrshDECZ7qL6qL-aoQzI_ZhPi1z5oe8kt2FCyNQa9fZagzrg_Tp6v0-_poreAi6oP-AExTsdqXe9E8UrWPULi8suvG9ItjFZOEgan6K214QTcXQiqrTRDrPrUeCXwfnCNS_W__mDuXS5GpKMt2yFS0dGC1nfStPkmloqjMoszIdY9FT33rLQdg0icjWsRXjDQUQaFyauqqg_YuuGQyq3qykUF_YZvh1srldBtBeGHAOptJZSK3qDe4eDwTQaxoLCEE9lJsvmiaIXR-6x-nchL1E_5feY5Iavx98DXF1Y-EW6KKLYoZcvndWXUFil9nc64OjwMu9wDE75-NKG0JtARST7a9PZYxIHsqKIARXAv94RTn30bXkpLlhgHNNWyZsCN-3QmB5E7G4DL5Z4S9_1KEPSLcp0cftKUhbUpeuNyDS1cvVJRMty-fB6CktjiwGR722UDqMrYJ2MQAXi4mO22qXRRA2Ha_BbmV_FlAlV60a2XprcjzNeGRbjR8l4SXRCpkv9fvVpwd4kz0vpVF6Bua7155RMG_7DzcWiPAxSzCrOrWi52VbR-3GXvLtkzi_u4jczBNvuVDrN_M9nOW2DJ44QbGUbdJunPeb43G1BYuc2oRDE8BdqpthS3sAwHbhu-kUBsiBfsvAz6F5_I0TMKaMBAT3RyctAyM_zBmFytrsjMTCYEvelChjjboqIfKXNsxqBH-P2gKC0l6K93_tvmCfNezqEBgOL0awX-OqYDDme_eaB5-UwzvmWNm-jOcv1Z8PM0RxJQ_MeyuEqarr3qtwexP93oIQBCgrxnrdFEDFczVoK8wkw-khfPPUn0Uj_f8CpDPhx9rtnfPQnyD_dJ-UQRrY_LFtzUGH4cP7UEKxatyaqihFtxAuhupyPpRn6nSk0zFbDrO265pXpC33Q_DgQpZRGJ08J-FqRg_s2wGgQ1WuXTghAMen5-2DHGrftaa77RE-biW5UUKteHTvr4_OUpkt8GFs_94oxtpBWH0qblI54uKfNdd9fLCaemkS69Oz8MED5VEOUe_GO4d2U2cSk8swizySpHb-hnDH6jBDiwm5GnJUYtL7H4oChBClNjUPaOGasrpd3LtIhgBZUg0DVd8DehXHpkRasxMP50gLRP4NCG-L15zVIAWjZZIe-4MUxj0eHHG9aRglnG4pTudg25_y3Tyo6PlH8FbLbKzFk6hodBk3uMy4fajhpb35JNPazDlkY74OY2WNsST85qctKZWzqK2plojEQJl6c5bjkAGAcOC8HXNwdKTvorRXFssIX6eNlg1jCKdBJuYzhcE1l6Qn7e5CVQWqYtTXUcfgrqPTHGoH1weUiISoQzuIXDUaygZoIhrtKpLu8_eeZTIApOjt3X7S7aEst3spPwJBvkDnp-AM4ussJDwB4bxbgP0pJkICY3V_vTg-HRhjiopeMdppo5qLyvHf9hxJCxBsL7vtVCNmsBlw6otMht-ItQuYbCNkTiqpHSKh843lQ-8P5EmFfMilwoD; __Secure-3PSIDTS=sidts-CjEBWhotCaRoXoNXQh_LpULm0aselCovonNz6w8uDiOhS_MBXfxyhPXJPHBAz75Ze2MCEAA; __Secure-3PSIDCC=AKEyXzWS60mAz2wOliq5lBo3YVICO72YOGF2beIf30MItp8injC0wemHt_WKXKE27iMOt2vNUpTrfw',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': 'e79c2a68-c9c9-4701-bda9-f3351fd299f4',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': 'd37cc422-d708-4174-9c1d-4253dac1dd5a',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_k2g8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_pfc8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_j7c8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_czd8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_j7c8446xgc',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 100,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'RSA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_wwga546xgc',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'may. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_tvr8446xgc',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_id_n3_',
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [15]:
rosario_renta = run_looker_pipeline(
    cookies, headers, params, json_data,
    pais="Argentina",
    ciudad="Rosario",
    operacion="rent",
    columnas={
        "col_0": "neighborhood",
        "col_1": "new",
        "col_2": "index",
        "col_3": "used"
    },
     mes_field_name="_fecha_"
)
print("Total de registros", len(rosario_renta))
rosario_renta.to_csv("rosario_renta.csv", encoding="latin1",index=False)
rosario_renta

📍 Usando filtro de mes en índice: 1
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,Puerto Norte,NaN,24632.0,24648.0,mar. 21,Argentina,Rosario,rent
1,Alberto Olmedo,21005.0,20399.0,20351.0,mar. 21,Argentina,Rosario,rent
2,Nuestra Sra. de Lourdes,21647.0,19700.0,19290.0,mar. 21,Argentina,Rosario,rent
3,Remedios Escalada de San Martín,21147.0,18662.0,17893.0,mar. 21,Argentina,Rosario,rent
4,Barrio del Abasto,NaN,18475.0,18294.0,mar. 21,Argentina,Rosario,rent
...,...,...,...,...,...,...,...,...
589,Alberdi,NaN,417962.0,NaN,ago. 25,Argentina,Rosario,rent
590,República de la 6 ta.,436010.0,405310.0,396410.0,ago. 25,Argentina,Rosario,rent
591,Domingo Faustino Sarmiento,NaN,388862.0,379100.0,ago. 25,Argentina,Rosario,rent
592,España y Hospitales,NaN,378359.0,368554.0,ago. 25,Argentina,Rosario,rent
